# Production Pipeline Demo

Demonstrates the two-stage RAG pipeline used in production:

1. **Stage 1 — Qwen retrieval**: encode the input paper with `qwen3-embedding`, compute cosine similarity against all BOUN paper embeddings, return top-20 candidate researchers.
2. **Stage 2 — Llama reranking**: send top-20 candidates to `llama3.2:3b` for LLM-based relevance reranking.

Requires Ollama running at `http://localhost:11434` with `qwen3-embedding:latest` and `llama3.2:3b` pulled.

In [ ]:
import sys
import os
import numpy as np
import requests
from sklearn.metrics.pairwise import cosine_similarity

sys.path.insert(0, 'src')
from similarity import ollama_rank_candidates

OLLAMA_URL = 'http://localhost:11434'
QWEN_MODEL = 'qwen3-embedding:latest'
LLAMA_MODEL = 'llama3.2:3b'
CACHE_PATH = 'data/embeddings_cache/boun_qwen_papers.npz'
TOP_K_STAGE1 = 20  # candidates to pass to Stage 2
TOP_K_FINAL  = 5   # final results to display

## 1. Load BOUN paper embeddings from cache

In [ ]:
assert os.path.exists(CACHE_PATH), f'Cache not found: {CACHE_PATH}. Run the backend seeder first.'

cache = np.load(CACHE_PATH, allow_pickle=True)
boun_embs      = cache['embeddings']          # (N, dim)
boun_r_ids     = cache['researcher_ids']      # (N,)  e.g. 'A5023..'
boun_paper_ids = cache['paper_ids']           # (N,)  OpenAlex paper IDs

print(f'Loaded {len(boun_embs):,} BOUN paper embeddings  (dim={boun_embs.shape[1]})')
print(f'Unique researchers: {len(set(boun_r_ids)):,}')

## 2. Define the input paper

Paste any external paper title + abstract here to test the pipeline.

In [ ]:
# Example: a deep-learning paper likely to match BOUN researchers
input_paper = {
    'title': 'Attention Is All You Need',
    'abstract': (
        'The dominant sequence transduction models are based on complex recurrent or '
        'convolutional neural networks that include an encoder and a decoder. '
        'The best performing models also connect the encoder and decoder through an attention mechanism. '
        'We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, '
        'dispensing with recurrence and convolutions entirely.'
    ),
    'concepts': 'transformer attention neural network sequence model NLP',
}

query_text = f"{input_paper['title']} {input_paper['abstract']} {input_paper['concepts']}"
print('Query text preview:', query_text[:200])

## 3. Stage 1 — Qwen embedding + cosine similarity retrieval

In [ ]:
def embed_qwen(texts: list[str], model: str = QWEN_MODEL, url: str = OLLAMA_URL) -> np.ndarray:
    """Encode texts via Ollama /api/embed."""
    batch = [t[:2048] for t in texts]
    resp = requests.post(f'{url}/api/embed', json={'model': model, 'input': batch}, timeout=120)
    resp.raise_for_status()
    return np.array(resp.json()['embeddings'], dtype=np.float32)

query_emb = embed_qwen([query_text])  # (1, dim)
print(f'Query embedding shape: {query_emb.shape}')

In [ ]:
# Cosine similarity against all BOUN papers
scores = cosine_similarity(query_emb, boun_embs)[0]  # (N,)

# Aggregate: mean score per researcher
from collections import defaultdict
r_scores: dict[str, list[float]] = defaultdict(list)
for r_id, score in zip(boun_r_ids, scores):
    r_scores[r_id].append(float(score))

r_mean = {r_id: np.mean(s) for r_id, s in r_scores.items()}
top_r_ids = sorted(r_mean, key=r_mean.get, reverse=True)[:TOP_K_STAGE1]

print(f'Stage 1 top-{TOP_K_STAGE1} researchers (by mean Qwen cosine similarity):')
for i, r_id in enumerate(top_r_ids, 1):
    print(f'  {i:2d}. {r_id}  score={r_mean[r_id]:.4f}')

## 4. Load researcher metadata from database (optional)

If the backend Postgres is running, we can fetch display names. Otherwise we'll use the OpenAlex IDs directly.

In [ ]:
# Try to fetch researcher metadata from the backend API
researcher_meta: dict[str, dict] = {}
try:
    resp = requests.get('http://localhost:8001/researchers', timeout=5)
    if resp.ok:
        for r in resp.json().get('researchers', []):
            researcher_meta[r['id']] = r
        print(f'Loaded metadata for {len(researcher_meta)} researchers from API')
    else:
        print('API returned non-200; using IDs only')
except Exception as e:
    print(f'Could not reach backend ({e}); using IDs only')

def display(r_id: str) -> str:
    return researcher_meta.get(r_id, {}).get('display_name', r_id)

## 5. Build candidate context for Stage 2

For each Stage 1 researcher, find their top-3 BOUN papers most similar to the query — these become the context for Llama.

In [ ]:
# Load paper texts from cache (we need the original text or rebuild from IDs)
# Since the .npz only has embeddings, we'll use the OpenAlex ID as the paper reference
# and build a concise candidate description for each researcher.

candidate_descriptions: list[str] = []
for r_id in top_r_ids:
    name = display(r_id)
    # Find indices for this researcher, sorted by similarity
    r_mask = np.where(boun_r_ids == r_id)[0]
    r_paper_scores = scores[r_mask]
    top3_local = r_mask[np.argsort(r_paper_scores)[::-1][:3]]
    top3_ids = boun_paper_ids[top3_local]
    top3_scores = scores[top3_local]
    papers_str = '; '.join(f'{pid} ({s:.3f})' for pid, s in zip(top3_ids, top3_scores))
    candidate_descriptions.append(
        f'Researcher: {name}  |  Top matching BOUN papers: {papers_str}'
    )

print('Candidate descriptions for Llama:')
for d in candidate_descriptions:
    print(' ', d[:120])

## 6. Stage 2 — Llama 3.2 3B reranking

In [ ]:
ranked = ollama_rank_candidates(
    query_text=query_text,
    candidates=candidate_descriptions,
    model_name=LLAMA_MODEL,
    ollama_url=OLLAMA_URL,
)

print(f'\nFinal top-{TOP_K_FINAL} researchers after Llama reranking:')
print(f'{"Rank":<5} {"Researcher":<40} {"Qwen score":<12} {"Llama score"}')
print('-' * 70)
for rank, (idx, llama_score) in enumerate(ranked[:TOP_K_FINAL], 1):
    r_id = top_r_ids[idx]
    print(f'{rank:<5} {display(r_id):<40} {r_mean[r_id]:<12.4f} {llama_score:.4f}')

## 7. Compare Stage 1 vs Stage 2 ordering

Shows how Llama reranking changes the order relative to pure Qwen cosine similarity.

In [ ]:
import pandas as pd

stage1_order = {r_id: i + 1 for i, r_id in enumerate(top_r_ids)}
rows = []
for llama_rank, (idx, llama_score) in enumerate(ranked, 1):
    r_id = top_r_ids[idx]
    rows.append({
        'Researcher': display(r_id),
        'Stage1 rank': stage1_order[r_id],
        'Stage1 score': round(r_mean[r_id], 4),
        'Stage2 rank': llama_rank,
        'Stage2 score': round(llama_score, 4),
        'Rank change': stage1_order[r_id] - llama_rank,
    })

df = pd.DataFrame(rows)
df.sort_values('Stage2 rank', inplace=True)
df.reset_index(drop=True, inplace=True)
df